In [ ]:
import trimesh
import pyvista as pv
print(pv.__version__) # Check PyVista version
import numpy as np
import matplotlib.pyplot as plt
import os
from vtkmodules.vtkRenderingCore import vtkCoordinate
# Assuming your cross-section functions are importable
# from calculate_cross_section_clean import get_section_points_at_plane # Hypothetical function
# from cross_section_functions import order_points
# --- ICP Alignment (Optional but Recommended) ---
try:
    from trimesh.registration import icp
    use_icp = True
except ImportError:
    print("Warning: 'open3d' or 'pycpd' not found. ICP alignment will be skipped.")
    use_icp = False
# --- End ICP ---


def compare_meshes(original_obj_path, idealised_ply_path, output_dir, base_name):
    """Generates comparison figures for original and idealised meshes, displayed side-by-side."""

    print(f"Comparing '{original_obj_path}' and '{idealised_ply_path}'")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        # 1. Load Meshes
        original_mesh = trimesh.load_mesh(original_obj_path)
        idealised_mesh = trimesh.load_mesh(idealised_ply_path)

        if isinstance(original_mesh, trimesh.Scene):
            original_mesh = original_mesh.dump(concatenate=True)
        if isinstance(idealised_mesh, trimesh.Scene):
            idealised_mesh = idealised_mesh.dump(concatenate=True)

        # --- Mesh Integrity Checks ---
        print(f"Original Mesh - Watertight: {original_mesh.is_watertight}")
        print(f"Idealised Mesh - Watertight: {idealised_mesh.is_watertight}")
        # Optional: Try processing to fix minor issues
        # original_mesh.process()
        # idealised_mesh.process()
        # --- End Checks ---

        # 2. Align Meshes
        # Start with centroid alignment
        center_orig = original_mesh.centroid
        center_ideal = idealised_mesh.centroid
        translation = center_orig - center_ideal
        if np.linalg.norm(translation) > 1e-4: # Only translate if significantly different
            print(f"  Aligning idealised mesh centroid to original ({translation})...")
            idealised_mesh.apply_translation(translation)

        # --- Advanced Alignment using ICP (Optional) ---
        if use_icp:
            try:
                # Sample points from both meshes for efficiency
                points_orig, _ = trimesh.sample.sample_surface(original_mesh, 2000)
                points_ideal, _ = trimesh.sample.sample_surface(idealised_mesh, 2000)

                # Run ICP
                icp_result = icp(points_ideal, points_orig, scale=False, max_iterations=100)

                transform_icp = None
                cost = None

                # Check the number of returned values and unpack carefully
                if isinstance(icp_result, (list, tuple)):
                    if len(icp_result) == 3:
                        transform_icp, cost, _ = icp_result
                    elif len(icp_result) == 2:
                        transform_icp, cost = icp_result
                    elif len(icp_result) == 1: # Maybe only transform is returned
                         transform_icp = icp_result[0]
                         cost = -1.0 # Indicate unknown cost
                    else:
                        print("  Warning: ICP returned an unexpected number of values.")
                        cost = -2.0 # Indicate error
                elif isinstance(icp_result, np.ndarray) and icp_result.shape == (4, 4):
                    # Handle case where only the transform matrix is returned
                    print("  Warning: ICP might have returned only the transformation matrix.")
                    transform_icp = icp_result
                    cost = -1.0 # Indicate unknown cost
                else:
                     print("  Warning: ICP returned an unknown format.")
                     cost = -3.0 # Indicate error

                # Validate transform_icp before proceeding
                if transform_icp is None or not isinstance(transform_icp, np.ndarray) or transform_icp.shape != (4, 4):
                    raise ValueError("Failed to retrieve a valid 4x4 transformation matrix from ICP.")

                # Process cost: Ensure it's a float before printing/using
                final_cost_str = "N/A"
                if cost is not None:
                    try:
                        # Attempt to convert cost to float, handle arrays
                        if isinstance(cost, np.ndarray):
                            if cost.size == 1:
                                cost_float = float(cost.item())
                            else:
                                # If cost is an array with multiple values, maybe take the mean or first?
                                print(f"  Warning: ICP cost is an array ({cost}), using first element.")
                                cost_float = float(cost.flat[0])
                        else:
                            cost_float = float(cost)

                        if np.isfinite(cost_float):
                             final_cost_str = f"{cost_float:.6f}"
                        else:
                             final_cost_str = "Invalid (non-finite)"
                    except (TypeError, ValueError) as cost_err:
                        print(f"  Warning: Could not interpret ICP cost value ({cost}): {cost_err}")
                        final_cost_str = f"Error ({cost})"

                print(f"  ICP alignment cost: {final_cost_str}")

                # Apply the found transformation to the idealised mesh
                idealised_mesh.apply_transform(transform_icp)
                print("  Applied ICP transformation to idealised mesh.")

            except Exception as e:
                # Print traceback for unexpected errors during ICP itself
                import traceback
                print(f"  Warning: Error during ICP alignment: {e}.")
                # traceback.print_exc() # Uncomment for full traceback if needed
                print("  Proceeding without ICP.")
        # --- End ICP Alignment ---


        # Convert to PyVista objects AFTER alignment
        original_pv = pv.wrap(original_mesh)
        idealised_pv = pv.wrap(idealised_mesh)

        # 3. Side-by-Side Visualization (PyVista)
        views = {'xy': 'top', 'xz': 'side'}
        # Define colors
        original_color = 'grey'
        idealised_color = '#D55E00' # Vermillion (colorblind-friendly)

        for view_plane, view_name in views.items():
            plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=[2048, 1024], border=False)

            # Subplot 1: Original Mesh
            plotter.subplot(0, 0)

            #plotter.add_text("Original Mesh", font_size=12)
            plotter.add_mesh(original_pv, color=original_color, smooth_shading=False, show_edges=False)
            if view_plane == 'xy': plotter.view_xy()
            elif view_plane == 'xz': plotter.view_xz()
            # plotter.camera.zoom(1.5) # Original zoom
            plotter.camera.zoom(1.2) # Reduced zoom

            # Subplot 2: Idealised Mesh
            plotter.subplot(0, 1)

            #plotter.add_text("Idealised Mesh", font_size=12)
            plotter.add_mesh(idealised_pv, color=idealised_color, smooth_shading=False, show_edges=False)
            if view_plane == 'xy': plotter.view_xy()
            elif view_plane == 'xz': plotter.view_xz()
            # plotter.camera.zoom(1.5) # Original zoom
            plotter.camera.zoom(1.2) # Reduced zoom

            plotter.link_views()

            p0 = np.array([10,10,0])
            p1 = p0 + np.array([5, 0, 0])
            
            coord = vtkCoordinate()
            coord.SetCoordinateSystemToWorld()
            coord.SetViewport(plotter.renderer)

            coord.SetValue(*p0)
            d0 = coord.GetComputedDoubleDisplayValue(plotter.renderer)
            coord.SetValue(*p1)
            d1 = coord.GetComputedDoubleDisplayValue(plotter.renderer)

            plotter.add_2d_line(d0, d1, color='black', width=5)
            mid = ((d0[0] + d1[0]) / 2, (d0[1] + d1[1]) / 2)
            plotter.add_2d_label(mid, "5 um", font_size=12, color='black')
            
            # img = plotter.screenshot(return_img=True)
            # import matplotlib.pyplot as plt
            # fig, ax = plt.subplots()
            # ax.imshow(img)
            # dx = 0.207566 # microns per pixel

            # from matplotlib_scalebar.scalebar import ScaleBar
            # scalebar = ScaleBar(
            #     dx = dx,
            #     units='um',
            #     length_fraction=0.2)
            # ax.add_artist(scalebar)
            # ax.axis('off')

            # Optional: Remove axes for cleaner look
            # plotter.remove_bounds_axes()

            output_path_3d = os.path.join(output_dir, f"{base_name}_comparison_3d_{view_name}_sidebyside.png")
            plotter.screenshot(output_path_3d, scale=3) # Increase scale for higher resolution
            print(f"  Saved side-by-side comparison ({view_name} view) to {output_path_3d}")
            plotter.close()

        # 4. 2D Cross-Section Comparison (Keep as before, if needed)
        # ... (Your existing 2D comparison code can remain here) ...


    except Exception as e:
        print(f"  Error processing comparison: {e}")
        import traceback
        traceback.print_exc()

# --- Example Usage ---
# Make sure this cell runs if you intend to execute the comparison
# (In Jupyter, you might run this cell directly)
# if __name__ == '__main__': # This check might not be necessary in a notebook cell
original_file = "Meshes/OBJ/Ac_DA_1_3.obj" # Example
idealised_file_std = "results/full_stomata_Ac_DA_1_3_std.ply" # Example
idealised_file_bulge = "results/full_stomata_Ac_DA_1_3_bulge.ply" # Example
comparison_output_dir = "results/comparisons"

base_name_orig = os.path.splitext(os.path.basename(original_file))[0]

if os.path.exists(original_file) and os.path.exists(idealised_file_std):
    compare_meshes(original_file, idealised_file_std, comparison_output_dir, f"{base_name_orig}_vs_std")

if os.path.exists(original_file) and os.path.exists(idealised_file_bulge):
    compare_meshes(original_file, idealised_file_bulge, comparison_output_dir, f"{base_name_orig}_vs_bulge")


0.44.2


ImportError: cannot import name 'vtkCoordinate' from 'vtkmodules.vtkCommonTransforms' (/usr/users/JIC_a5/tomkinsm/anaconda3/lib/python3.11/site-packages/vtkmodules/vtkCommonTransforms.cpython-311-x86_64-linux-gnu.so)